In [85]:
import pandas as pd
import numpy as np

In [86]:
df = pd.read_json("./rf_data/peters2.json")

X = df[["dx", "dbx", "max_v", "da1", "da2", "dt", "curr_floor"]]
y = df["new_floor"]

In [87]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [88]:
from sklearn.ensemble import RandomForestClassifier

# Create the Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=87,   # Number of trees
    random_state=42,     # Random seed
    bootstrap=False,
    max_features='log2',
    min_samples_leaf=2,
    min_samples_split=5
)

# Train the model
rf_model.fit(X_train, y_train)

RandomForestClassifier(bootstrap=False, max_features='log2', min_samples_leaf=2,
                       min_samples_split=5, n_estimators=87, random_state=42)

In [89]:
from sklearn.metrics import accuracy_score

# Predict on the test set
y_pred = rf_model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 100.00%


In [90]:
# Number of trees in random forest
n_estimators = [int(x) for x in np.linspace(start=50, stop=200, num=5)]  # Wider range
# Number of features to consider at every split
max_features = ['sqrt', 'log2']  # Added 'log2' for completeness
# Maximum number of levels in tree
max_depth = [2, 4, 6, 8, None]  # Expanded range
# Minimum number of samples required to split a node
min_samples_split = [2, 5, 10]  # Added a higher value
# Minimum number of samples required at each leaf node
min_samples_leaf = [1, 2, 4]  # Added another value
# Method of selecting samples for training each tree
bootstrap = [True, False]

param_grid = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf,
               'bootstrap': bootstrap}

In [91]:
rf_Model = RandomForestClassifier()

In [92]:
from sklearn.model_selection import GridSearchCV
rf_Grid = GridSearchCV(estimator = rf_Model, param_grid = param_grid, cv = 3, verbose=2, n_jobs = 4)

In [93]:
rf_Grid.fit(X_train, y_train)

Fitting 3 folds for each of 900 candidates, totalling 2700 fits


KeyboardInterrupt: 

In [82]:
rf_Grid.best_params_

{'bootstrap': True,
 'max_depth': 4,
 'max_features': 'log2',
 'min_samples_leaf': 2,
 'min_samples_split': 2,
 'n_estimators': 87}

In [ ]:
probs = rf_Grid.predict_proba(X_test)  

# Compute the confidence for each prediction as the maximum class probability
confidence = probs.max(axis=1)

# Average confidence across all predictions
avg_confidence = confidence.mean()

print (f'Train Accuracy - : {rf_Grid.score(X_train,y_train):.3f}')
print (f'Test Accuracy - : {rf_Grid.score(X_test,y_test):.3f}')
print(X_test)
for i in confidence:
    print(f'Confidence on Test Data: {i:.3f}')

Train Accuracy - : 1.000
Test Accuracy - : 1.000
          dx       dbx     max_v       da1       da2      dt  curr_floor
27 -1.638511 -3.052465  0.319683 -0.128362  0.279286  14.255           1
40  2.655421  5.042353  0.272660  0.177467 -0.266680  19.462           0
26 -1.667620 -2.462476  0.323701 -0.126671  0.289309  15.087           2
43  0.991844  2.276354  0.233402  0.111761 -0.281957  14.255           1
24  1.034008  2.838904  0.245932  0.110520 -0.255958  13.520           0
37  1.358945  2.927671  0.267085  0.170692 -0.263276  14.359           1
12  1.213588  2.862827  0.281073  0.184037 -0.250490  13.735           0
19  1.042607  2.710058  0.225952  0.105354 -0.257438  14.463           1
4   2.818242  6.457933  0.286142  0.181184 -0.256263  19.143           0
25  1.347528  2.110835  0.258157  0.181081 -0.252124  14.463           1
Confidence on Test Data: 0.727
Confidence on Test Data: 0.932
Confidence on Test Data: 0.681
Confidence on Test Data: 0.577
Confidence on Test Data:

In [ ]:
import joblib

# Save the trained model
joblib.dump(rf_Grid, 'random_forest_model.pkl')
print("Model saved as 'random_forest_model.pkl'")

Model saved as 'random_forest_model.pkl'
